<a href="https://colab.research.google.com/github/Ginkno/postech-tech-challenge-fase2-grupo1-12dtat/blob/Elizabeth/notebooks/02_preprocessamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — Pré-processamento

**Dimensão 4 da rúbrica — 15 pontos.**

Cada decisão precisa de justificativa escrita. Decidir *não* criar features é
aceitável, desde que o motivo esteja explícito.

In [136]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

RAW_1 = Path("../data/raw/application_record.csv")
RAW_2 = Path("../data/raw/credit_record.csv")

PROCESSED = Path("../data/processed")
TARGET = "IS_BAD_PAYER"


pd.set_option("display.max_columns", None)

In [137]:
#Leitura do dataframe e visualização das cinco primeiras linhas.

df_application = pd.read_csv(RAW_1)
df_application.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0


## 1. Dados faltantes

In [138]:
#Verificando a existência de dados nulos.

df_application.isnull().sum()

ID                          0
CODE_GENDER                 0
FLAG_OWN_CAR                0
FLAG_OWN_REALTY             0
CNT_CHILDREN                0
AMT_INCOME_TOTAL            0
NAME_INCOME_TYPE            0
NAME_EDUCATION_TYPE         0
NAME_FAMILY_STATUS          0
NAME_HOUSING_TYPE           0
DAYS_BIRTH                  0
DAYS_EMPLOYED               0
FLAG_MOBIL                  0
FLAG_WORK_PHONE             0
FLAG_PHONE                  0
FLAG_EMAIL                  0
OCCUPATION_TYPE        134203
CNT_FAM_MEMBERS             0
dtype: int64

Verificando a existência de dados nulos e como resultado trouxe que a coluna OCCUPATION_TYPE possui 134203 linhas em branco.

In [139]:
#Substituir os dados em branco da variável OCCUPATION_TYPE por "Unknown".

df_application["OCCUPATION_TYPE"] = df_application["OCCUPATION_TYPE"].fillna("Unknown")

**Decisão:** Para que não haja dados em branco na coluna OCCUPATION_TYPE substituímos por 'Unknow'.

In [140]:
#Converter dias de nascimento para idade em anos e para inteiro.
df_application["YEARS_BIRTH"] = (df_application["DAYS_BIRTH"] / -365).astype(int)

#Criar a coluna IS_INACTIVE (1 para inativo, 0 para empregado).
df_application["IS_INACTIVE"] = (df_application["DAYS_EMPLOYED"] == 365243).astype(int)

#Converter dias de emprego para anos.
df_application["YEARS_EMPLOYED"] = (df_application["DAYS_EMPLOYED"] / -365 ).astype(int)

In [141]:
# Contar quantas vezes o valor -1000 aparece em YEARS_EMPLOYED
count_minus_1000 = (df_application['YEARS_EMPLOYED'] == -1000).sum()
print(f"Quantidade de registros com YEARS_EMPLOYED == -1000: {count_minus_1000}")

# Verificar a relação entre YEARS_EMPLOYED == -1000 e IS_INACTIVE
print("Verificando se -1000 em YEARS_EMPLOYED corresponde a IS_INACTIVE == 1:")
display(df_application.loc[df_application['YEARS_EMPLOYED'] == -1000, 'IS_INACTIVE'].value_counts())

Quantidade de registros com YEARS_EMPLOYED == -1000: 75329
Verificando se -1000 em YEARS_EMPLOYED corresponde a IS_INACTIVE == 1:


IS_INACTIVE
1    75329
Name: count, dtype: int64

Como esperado, o valor -1000 em YEARS_EMPLOYED corresponde exatamente aos registros onde IS_INACTIVE é 1, ou seja, aos 'pensionistas/inativos'. Para facilitar a análise e a modelagem, substituiremos esse valor por 0, indicando zero anos de emprego ativo, o que é consistente com a definição de inatividade.

In [142]:
# Substituir -1000 em YEARS_EMPLOYED por 0
df_application['YEARS_EMPLOYED'] = df_application['YEARS_EMPLOYED'].replace(-1000, 0)

# Verificar as primeiras linhas após a substituição
print("df_application.head() após substituição de -1000 por 0:")
display(df_application.head())

df_application.head() após substituição de -1000 por 0:


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,32,0,12
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,Unknown,2.0,32,0,12
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,58,0,3
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,52,0,8
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,52,0,8


In [143]:
df_application = df_application.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED'])
display(df_application.head())

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,58,0,3
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8


In [144]:
#Leitura do dataframe e visualização das cinco primeiras linhas.
df_credit = pd.read_csv(RAW_2)
df_credit.head()

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C


Conforme dicionário de dados disponibilizado: 0: 1-29 dias atrasados; 1: 30-59 dias atrasados; 2: 60-89 dias atrasados; 3: 90-119 dias atrasados; 4: 120-149 dias atrasados; 5: dívidas atrasadas ou incobráveis, perdas por mais de 150 dias; C: quitadas naquele mês; X: sem empréstimo no mês;

In [145]:
#Verificando a existência de dados nulos.

df_credit.isnull().sum()

ID                0
MONTHS_BALANCE    0
STATUS            0
dtype: int64

Como resultado trouxe que a quantidade de dados nulos é 0.

Para unir as informações de df_credit com as de df_application, é preciso consolidar o histórico de crédito de cada cliente (ID) em uma única linha.

Codificar a variável STATUS: Atribuir valores numéricos que reflitam o risco de crédito. Agregar por ID: Para cada cliente, obter a pontuação de risco máximo (pior status) e o número de meses no histórico (MONTHS_BALANCE).

In [146]:
# Criar uma cópia de df_credit para evitar modificar o original
df_status_history = df_credit.copy()

# Consolidação do histórico de crédito. STATUS_LATED: 0 (1_29 dias) até o 5 (atraso grave).
#STATUS_ON_TIME: C (pago em dia).
#STATUS_NO_CREDIT: X (sem empréstimo no mês).
df_status_history["is_lated"] = df_status_history["STATUS"].isin(["0","1", "2", "3", "4", "5"]).astype(int)
df_status_history["is_on_time"] = (df_status_history["STATUS"] =="C").astype(int)
df_status_history["is_no_credit"] = (df_status_history["STATUS"] =="X").astype(int)

#Mapeamento do Risco: C e X não tem rsico(0). O status "0" inicia a escala de atraso (1).
risk_mapping = {
    "C": 0,
    "X": 0,
    "0": 1, "1": 2, "2": 3, "3": 4, "4": 5, "5": 6}
df_status_history["risk_score"] = df_status_history["STATUS"].map(risk_mapping)

#Reduzir para uma linha por ID.
df_status_aggregated = df_status_history.groupby("ID").agg(
    TOTAL_MOUNTHS_OBSERVED =("MONTHS_BALANCE", "count"),
    STATUS_LATED = ("is_lated", "sum"),
    STATUS_ON_TIME = ("is_on_time", "sum"),
    STATUS_NO_CREDIT = ("is_no_credit", "sum"),
    WORST_LATE_HISTORY = ("risk_score", "max")
    ).reset_index()

print("Histórico consolidado com sucesso (Uma linha por ID):")
print(df_status_aggregated.head())

Histórico consolidado com sucesso (Uma linha por ID):
        ID  TOTAL_MOUNTHS_OBSERVED  STATUS_LATED  STATUS_ON_TIME  \
0  5001711                       4             3               0   
1  5001712                      19            10               9   
2  5001713                      22             0               0   
3  5001714                      15             0               0   
4  5001715                      60             0               0   

   STATUS_NO_CREDIT  WORST_LATE_HISTORY  
0                 1                   1  
1                 0                   1  
2                22                   0  
3                15                   0  
4                60                   0  


In [147]:
# Mesclar df_application com df_status_aggregated usando a coluna 'ID'
df_final = pd.merge(df_application, df_status_aggregated, on='ID', how='inner')

# Exibir as primeiras linhas do dataframe final
print("Primeiras 5 linhas do df_final:")
display(df_final.head())

# Exibir o formato do dataframe final
print(f"Dimensão do df_final: {df_final.shape}")

Primeiras 5 linhas do df_final:


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED,TOTAL_MOUNTHS_OBSERVED,STATUS_LATED,STATUS_ON_TIME,STATUS_NO_CREDIT,WORST_LATE_HISTORY
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12,16,2,13,1,2
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12,15,2,12,1,2
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,58,0,3,30,7,7,16,1
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8,5,2,0,3,1
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8,5,0,0,5,0


Dimensão do df_final: (36457, 24)


In [148]:
# Visualizar a distribuição numérica da variável 'WORST_LATE_HISTORY'
print("Distribuição da variável 'WORST_LATE_HISTORY':")
display(df_final['WORST_LATE_HISTORY'].value_counts().sort_index())

Distribuição da variável 'WORST_LATE_HISTORY':


WORST_LATE_HISTORY
0     4455
1    27711
2     3675
3      314
4       76
5       46
6      180
Name: count, dtype: int64

In [149]:
#Verificando a existência de dados nulos.

df_final.isnull().sum()

ID                        0
CODE_GENDER               0
FLAG_OWN_CAR              0
FLAG_OWN_REALTY           0
CNT_CHILDREN              0
AMT_INCOME_TOTAL          0
NAME_INCOME_TYPE          0
NAME_EDUCATION_TYPE       0
NAME_FAMILY_STATUS        0
NAME_HOUSING_TYPE         0
FLAG_MOBIL                0
FLAG_WORK_PHONE           0
FLAG_PHONE                0
FLAG_EMAIL                0
OCCUPATION_TYPE           0
CNT_FAM_MEMBERS           0
YEARS_BIRTH               0
IS_INACTIVE               0
YEARS_EMPLOYED            0
TOTAL_MOUNTHS_OBSERVED    0
STATUS_LATED              0
STATUS_ON_TIME            0
STATUS_NO_CREDIT          0
WORST_LATE_HISTORY        0
dtype: int64

## 2. Definição da variável alvo

_Se houve binarização, qual limiar e por quê? Justifique com base na distribuição, não por convenção._

In [150]:
# Criar a variável alvo binária IS_BAD_PAYER
# 1 se WORST_LATE_HISTORY > 1 (mau pagador),
# 0 caso contrário (bom pagador ou sem atrasos significativos)

df_final["IS_BAD_PAYER"] = (df_final["WORST_LATE_HISTORY"] > 1).astype(int)

print("Contagem de 'bons' e 'maus' pagadores:")
display(df_final['IS_BAD_PAYER'].value_counts())

print("Proporção de 'bons' e 'maus' pagadores:")
display(df_final["IS_BAD_PAYER"].value_counts(normalize=True).mul(100).apply(lambda x: f'{x:.2f}%'))

Contagem de 'bons' e 'maus' pagadores:


IS_BAD_PAYER
0    32166
1     4291
Name: count, dtype: int64

Proporção de 'bons' e 'maus' pagadores:


IS_BAD_PAYER
0    88.23%
1    11.77%
Name: proportion, dtype: object

A variável IS_BAD_PAYER foi criada a partir da variável WORST_LATE_HISTORY. O limiar escolhido foi **1**. Clientes com WORST_LATE_HISTORY > 1 (ou seja, aqueles que tiveram atrasos de 30 dias ou mais, correspondente ao STATUS '1' ou superior) foram classificados como 'maus pagadores' (1), enquanto aqueles com WORST_LATE_HISTORY <= 1` (ou seja, sem atrasos significativos, STATUS 'X', 'C', ou '0') foram classificados como 'bons pagadores' (0).

Essa escolha é justificada pela severidade do atraso no crédito. Um WORST_LATE_HISTORY de 1 corresponde a um atraso de 1 a 29 dias. A partir de WORST_LATE_HISTORY igual ou superior a 2 o cliente já apresentou um atraso de 30 a 59 dias ou mais, o que é um indicador mais forte de risco de crédito. A distribuição numérica acima ilustra a contagem de clientes em cada nível de WORST_LATE_HISTORY e como o limiar de  `> 1` separa os dois grupos.

Com a criação da variável IS_BAD_PAYER, podemos observar que a maioria dos clientes é classificada como 'bom pagador' (0), enquanto uma minoria significativa é classificada como 'mau pagador' (1).

In [151]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36457 entries, 0 to 36456
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ID                      36457 non-null  int64  
 1   CODE_GENDER             36457 non-null  object 
 2   FLAG_OWN_CAR            36457 non-null  object 
 3   FLAG_OWN_REALTY         36457 non-null  object 
 4   CNT_CHILDREN            36457 non-null  int64  
 5   AMT_INCOME_TOTAL        36457 non-null  float64
 6   NAME_INCOME_TYPE        36457 non-null  object 
 7   NAME_EDUCATION_TYPE     36457 non-null  object 
 8   NAME_FAMILY_STATUS      36457 non-null  object 
 9   NAME_HOUSING_TYPE       36457 non-null  object 
 10  FLAG_MOBIL              36457 non-null  int64  
 11  FLAG_WORK_PHONE         36457 non-null  int64  
 12  FLAG_PHONE              36457 non-null  int64  
 13  FLAG_EMAIL              36457 non-null  int64  
 14  OCCUPATION_TYPE         36457 non-null

## 3. Normalização / padronização

**Escolha e justificativa:** _StandardScaler, MinMaxScaler ou nenhum?_

Aplique preferencialmente dentro de um `Pipeline` no notebook 03 — ajustar o scaler antes do split vaza informação do teste para o treino.

**Escolha e Justificativa:** `StandardScaler`

Para este projeto de classificação de risco de crédito, bom ou mau pagador,a escolha foi utilizar o `StandardScaler`. O conjunto de dados apresenta variáveis com magnitudes muito distintas e a presença de alguns outliers foi mantida.

**Justificativas para o `StandardScaler`:**

1.  **Robustez a Outliers:** O `StandardScaler` reescala os dados para que tenham média zero e desvio padrão um. Embora os outliers ainda possam afetar a média e o desvio padrão, o `StandardScaler` é, em geral, **menos sensível a outliers do que o `MinMaxScaler`**. Se usássemos o `MinMaxScaler`, que reescala os dados para um intervalo fixo (e.g., [0,1]) usando os valores mínimos e máximos observados, a presença de outliers extremos poderia comprimir a maioria dos dados em uma faixa muito pequena desse intervalo, diminuindo a variabilidade útil das features. O `StandardScaler`, por não limitar os dados a um intervalo fixo, permite que os outliers mantenham sua distância relativa, o que pode ser benéfico dependendo do algoritmo.

2.  **Generalidade e Ampla Aplicabilidade:** Muitos algoritmos de Machine Learning (como Regressão Linear, Regressão Logística, SVMs, Redes Neurais, K-Means e PCA) funcionam melhor ou convergem mais rapidamente quando as features estão padronizadas para terem média zero e variância unitária.

3.  **Preservação de Informações da Distribuição:** Ao centralizar os dados em zero e escalá-los pela variância, o `StandardScaler` ajuda a mitigar o impacto de diferentes escalas sem distorcer significativamente a forma da distribuição original dos dados (ao contrário de transformações não-lineares, que alteram a distribuição). Isso é importante para algoritmos que se beneficiam da estrutura da distribuição dos dados.

**Por que não `MinMaxScaler`?**

Embora o `MinMaxScaler` seja útil para algoritmos que exigem entradas em um intervalo estritamente definido (como algumas redes neurais com funções de ativação específicas), sua alta sensibilidade a outliers o torna menos ideal neste cenário, onde decidimos manter alguns desses valores extremos. Um único outlier pode distorcer drasticamente a escala, fazendo com que a maioria dos dados fique 'espremida' em uma pequena parte do novo intervalo.

**Implementação no `Pipeline` (Notebook 03):**

Conforme mencionado, é crucial aplicar o `StandardScaler` corretamente para evitar **vazamento de dados (data leakage)**. O processo de 'ajuste' (fit) do scaler (onde ele calcula a média e o desvio padrão) deve ser realizado **exclusivamente no conjunto de dados de treino**. Posteriormente, o scaler 'treinado' será usado para 'transformar' tanto os dados de treino quanto os dados de teste.

Para garantir essa separação rigorosa e automatizar o processo, o `StandardScaler` será integrado a um `Pipeline` no Notebook 03, conforme orientado no enunciado dessa etapa 03. Isso assegura que nenhuma informação do conjunto de teste contamine o processo de reescalonamento, resultando em uma avaliação mais honesta e precisa da performance do modelo.

## 4. Feature engineering

**Justificativa:** Para enriquecer o modelo e capturar relações mais complexas nos dados, foram criadas as seguintes novas features:

INCOME_PER_PERSON (Renda por Pessoa da Família):

Cálculo: AMT_INCOME_TOTAL / CNT_FAM_MEMBERS
Justificativa: A renda total (AMT_INCOME_TOTAL) é um fator importante, mas a capacidade de um indivíduo de honrar seus compromissos financeiros pode ser mais precisamente avaliada ao considerar quantas pessoas dependem dessa renda. Uma renda alta distribuída entre muitos membros da família pode indicar menor folga financeira por indivíduo, enquanto a mesma renda para uma família menor sugere maior capacidade de pagamento. Esta feature normaliza a renda pelo tamanho da família, oferecendo uma perspectiva de renda per capita que pode ser um indicador mais robusto de estabilidade financeira e risco de crédito.

HAS_CHILDREN (Possui Filhos):

Cálculo: Binário (1 se CNT_CHILDREN > 0, 0 caso contrário)
Justificativa: A presença de filhos (CNT_CHILDREN > 0) pode influenciar significativamente as responsabilidades financeiras de um indivíduo. Famílias com filhos geralmente têm despesas mais elevadas, o que pode impactar sua capacidade de pagamento de dívidas. Embora CNT_CHILDREN já exista, uma feature binária explícita pode simplificar a interpretação para alguns modelos e destacar a diferença fundamental entre ter ou não ter filhos, independentemente do número exato de crianças.

PROP_LATED (Proporção de Meses com Atraso):

Cálculo: STATUS_LATED / TOTAL_MOUNTHS_OBSERVED
Justificativa: Esta feature indica a frequência relativa de atrasos no pagamento em relação ao total de meses observados no histórico de crédito do cliente. Uma alta proporção de atrasos pode ser um forte indicador de risco de crédito, independentemente da gravidade máxima do atraso. Ela oferece uma visão mais granular e temporal do comportamento de inadimplência.

PROP_ON_TIME (Proporção de Meses Pagos em Dia):

Cálculo: STATUS_ON_TIME / TOTAL_MOUNTHS_OBSERVED
Justificativa: Complementar à proporção de atrasos, esta feature mede a frequência com que o cliente pagou suas obrigações em dia. Uma alta proporção de pagamentos em dia é um bom sinal de adimplência e responsabilidade financeira, o que pode mitigar o impacto de atrasos pontuais e contribuir para uma avaliação de crédito mais equilibrada.

In [152]:
# Criar features de proporcao do historico de credito.
df_final['INCOME_PER_PERSON'] = df_final['AMT_INCOME_TOTAL'] / df_final['CNT_FAM_MEMBERS']
df_final['HAS_CHILDREN'] = (df_final['CNT_CHILDREN'] > 0).astype(int)
df_final['PROP_LATED'] = df_final['STATUS_LATED'] / df_final['TOTAL_MOUNTHS_OBSERVED']
df_final['PROP_ON_TIME'] = df_final['STATUS_ON_TIME'] / df_final['TOTAL_MOUNTHS_OBSERVED']

In [153]:
df_final.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED,TOTAL_MOUNTHS_OBSERVED,STATUS_LATED,STATUS_ON_TIME,STATUS_NO_CREDIT,WORST_LATE_HISTORY,IS_BAD_PAYER,INCOME_PER_PERSON,HAS_CHILDREN,PROP_LATED,PROP_ON_TIME
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12,16,2,13,1,2,1,213750.0,0,0.125000,0.812500
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,1,1,0,0,Unknown,2.0,32,0,12,15,2,12,1,2,1,213750.0,0,0.133333,0.800000
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,1,0,0,0,Security staff,2.0,58,0,3,30,7,7,16,1,0,56250.0,0,0.233333,0.233333
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8,5,2,0,3,1,0,270000.0,0,0.400000,0.000000
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,1,0,1,1,Sales staff,1.0,52,0,8,5,0,0,5,0,0,270000.0,0,0.000000,0.000000


## Tratamento de variáveis categóricas

As colunas categóricas detectadas em `df_processed_x` são codificadas individualmente com `LabelEncoder`, gerando um número inteiro para cada categoria. Esses inteiros são identificadores de classe, não pesos definidos manualmente.

Atenção: alguns modelos interpretam esses números como uma ordem ou distância (por exemplo, podem tratar a categoria 2 como maior que a categoria 1). Para variáveis nominais sem ordem natural, use `OneHotEncoder` no pipeline de modelagem se for necessário evitar essa interpretação.

In [154]:
# Usando um LabelEncoder independente para cada coluna categórica.
from sklearn.preprocessing import LabelEncoder

df_processed_x = df_final.copy()

df_processed_x = df_processed_x[[c for c in df_processed_x if c not in ['IS_BAD_PAYER']] + ['IS_BAD_PAYER']]

categorical_cols = df_processed_x.select_dtypes(include=["object", "category", "bool"]).columns
label_encoders = {}

for col in categorical_cols:
    encoder = LabelEncoder()
    df_processed_x[col] = encoder.fit_transform(df_processed_x[col].astype("string").fillna("__MISSING__"))
    label_encoders[col] = encoder

print("Colunas categóricas codificadas:", list(categorical_cols))
display(df_processed_x.head())

Colunas categóricas codificadas: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']


,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED,TOTAL_MOUNTHS_OBSERVED,STATUS_LATED,STATUS_ON_TIME,STATUS_NO_CREDIT,WORST_LATE_HISTORY,INCOME_PER_PERSON,HAS_CHILDREN,PROP_LATED,PROP_ON_TIME,IS_BAD_PAYER
0,5008804,1,1,1,0,427500.0,4,1,0,4,1,1,0,0,17,2.0,32,0,12,16,2,13,1,2,213750.0,0,0.125000,0.812500,1
1,5008805,1,1,1,0,427500.0,4,1,0,4,1,1,0,0,17,2.0,32,0,12,15,2,12,1,2,213750.0,0,0.133333,0.800000,1
2,5008806,1,1,1,0,112500.0,4,4,1,1,1,0,0,0,16,2.0,58,0,3,30,7,7,16,1,56250.0,0,0.233333,0.233333,0
3,5008808,0,0,1,0,270000.0,0,4,3,1,1,0,1,1,14,1.0,52,0,8,5,2,0,3,1,270000.0,0,0.400000,0.000000,0
4,5008809,0,0,1,0,270000.0,0,4,3,1,1,0,1,1,14,1.0,52,0,8,5,0,0,5,0,270000.0,0,0.000000,0.000000,0


In [155]:
# Fazer uma cópia do Dataframe.
df_processed = df_final.copy()

# Mapear variáveis categóricas binárias para numéricas.
df_processed["CODE_GENDER"] = df_processed["CODE_GENDER"].map({"M": 1, "F": 0})
df_processed["FLAG_OWN_CAR"] = df_processed["FLAG_OWN_CAR"].map({"Y": 1, "N": 0})
df_processed["FLAG_OWN_REALTY"] = df_processed["FLAG_OWN_REALTY"].map({"Y": 1, "N": 0})

# Identificar colunas categóricas restantes que não são binárias para One-Hot Encoding
categorical_cols_to_encode = [
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE'
]

# Aplicar One-Hot Encoding para as variáveis categóricas restantes
# e juntar as colunas codificadas ao df_processed, removendo as colunas originais.
df_final_encoded = pd.get_dummies(df_processed, columns=categorical_cols_to_encode, prefix=categorical_cols_to_encode, drop_first=True)

#Remover a coluna 'ID' já que não é relevante para a previsão,
#pois é apenas um identificador único.
df_final_encoded = df_final_encoded.drop(columns=['ID'])
print("df_final_encoded após o tratamento das variáveis categóricas:")
display(df_final_encoded.head())

df_final_encoded após o tratamento das variáveis categóricas:


,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,CNT_FAM_MEMBERS,YEARS_BIRTH,IS_INACTIVE,YEARS_EMPLOYED,TOTAL_MOUNTHS_OBSERVED,STATUS_LATED,STATUS_ON_TIME,STATUS_NO_CREDIT,WORST_LATE_HISTORY,IS_BAD_PAYER,INCOME_PER_PERSON,HAS_CHILDREN,PROP_LATED,PROP_ON_TIME,NAME_INCOME_TYPE_Pensioner,NAME_INCOME_TYPE_State servant,NAME_INCOME_TYPE_Student,NAME_INCOME_TYPE_Working,NAME_EDUCATION_TYPE_Higher education,NAME_EDUCATION_TYPE_Incomplete higher,NAME_EDUCATION_TYPE_Lower secondary,NAME_EDUCATION_TYPE_Secondary / secondary special,NAME_FAMILY_STATUS_Married,NAME_FAMILY_STATUS_Separated,NAME_FAMILY_STATUS_Single / not married,NAME_FAMILY_STATUS_Widow,NAME_HOUSING_TYPE_House / apartment,NAME_HOUSING_TYPE_Municipal apartment,NAME_HOUSING_TYPE_Office apartment,NAME_HOUSING_TYPE_Rented apartment,NAME_HOUSING_TYPE_With parents,OCCUPATION_TYPE_Cleaning staff,OCCUPATION_TYPE_Cooking staff,OCCUPATION_TYPE_Core staff,OCCUPATION_TYPE_Drivers,OCCUPATION_TYPE_HR staff,OCCUPATION_TYPE_High skill tech staff,OCCUPATION_TYPE_IT staff,OCCUPATION_TYPE_Laborers,OCCUPATION_TYPE_Low-skill Laborers,OCCUPATION_TYPE_Managers,OCCUPATION_TYPE_Medicine staff,OCCUPATION_TYPE_Private service staff,OCCUPATION_TYPE_Realty agents,OCCUPATION_TYPE_Sales staff,OCCUPATION_TYPE_Secretaries,OCCUPATION_TYPE_Security staff,OCCUPATION_TYPE_Unknown,OCCUPATION_TYPE_Waiters/barmen staff
0,1,1,1,0,427500.0,1,1,0,0,2.0,32,0,12,16,2,13,1,2,1,213750.0,0,0.125000,0.812500,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
1,1,1,1,0,427500.0,1,1,0,0,2.0,32,0,12,15,2,12,1,2,1,213750.0,0,0.133333,0.800000,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
2,1,1,1,0,112500.0,1,0,0,0,2.0,58,0,3,30,7,7,16,1,0,56250.0,0,0.233333,0.233333,False,False,False,True,False,False,False,True,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
3,0,0,1,0,270000.0,1,0,1,1,1.0,52,0,8,5,2,0,3,1,0,270000.0,0,0.400000,0.000000,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False
4,0,0,1,0,270000.0,1,0,1,1,1.0,52,0,8,5,0,0,5,0,0,270000.0,0,0.000000,0.000000,False,False,False,False,False,False,False,True,False,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False


Considerações Finais sobre Pré-processamento: Outliers e Balanceamento de Classes

É importante ressaltar que, nesta etapa de pré-processamento (Notebook 02), algumas técnicas comuns não foram aplicadas, especificamente:

*   **Tratamento de Outliers (Capping):** A decisão de não aplicar o capping para outliers neste momento é estratégica. Embora a presença de valores extremos possa influenciar alguns modelos, a remoção ou transformação precoce de outliers pode levar à perda de informações valiosas ou à criação de um viés artificial, especialmente antes da divisão dos dados em conjuntos de treino e teste. Além disso, a aplicação de técnicas de capping é mais robusta quando realizada dentro de um `Pipeline` de pré-processamento no notebook 03, garantindo que o ajuste dos limites de capping ocorra apenas nos dados de treino, evitando assim o **vazamento de dados (data leakage)** do conjunto de teste. (Obs.: os outliers foram tratados na etapa "04 Outliers" do notebook 01, para responder tal tópico do trabalho, porém os códigos não foram trazidos para esse notebook 02.)  

*   **Balanceamento de Classes:** O conjunto de dados apresenta um desequilíbrio de classes significativo para a variável alvo `IS_BAD_PAYER`, conforme observado na seção de definição da variável alvo. No entanto, o balanceamento de classes (usando técnicas como oversampling, undersampling ou SMOTE) não foi realizado nesta fase. A principal razão para isso é evitar o **vazamento de dados**. Técnicas de balanceamento, especialmente oversampling, criam novas amostras baseadas nos dados existentes. Se aplicadas antes da divisão treino-teste, as amostras sintéticas criadas podem "ver" informações do conjunto de teste, levando a uma superestimação da performance do modelo.

Ambas as medidas — o tratamento de outliers (se considerado necessário após análise mais aprofundada) e o balanceamento de classes — serão consideradas e implementadas no **Notebook 03**, dentro de um `Pipeline` de pré-processamento. Isso garantirá que todas as transformações de dados sejam aplicadas de maneira apropriada, prevenindo o vazamento de informações e resultando em uma avaliação de modelo mais justa e confiável.

## 5. Salvar dataset tratado

In [156]:
df_final_encoded.to_csv(PROCESSED / "dataset_tratado.csv", index=False)
df_processed_x.to_csv(PROCESSED / "dataset_processed_x.csv", index=False)